# 06 — Domain Adaptation: Few-Shot Target + Cross-Dataset Mixup

**Pertanyaan inti:** dari temuan T6 (joint training ~in-domain; gap = *distribution shift*), berapa **sedikit** label dari jaringan target yang dibutuhkan agar NIDS cross-network berfungsi? Dan bisakah **mixup lintas-dataset** membantu tanpa label target?

**Dua eksperimen (Model A, 9 fitur, biner, z-score per dataset):**
1. **Few-shot target adaptation** — latih di *source* + fraksi kecil label *target* (0%, 1%, 5%, 10%, 25%), uji di sisa *target*. Kurva MCC vs fraksi label target. Dua arah (CIC→UNSW, UNSW→CIC). Menjawab pertanyaan deployment nyata: berapa kalibrasi minimum jaringan baru.
2. **Cross-dataset mixup** — sampel sintetik `x = λ·x_src + (1−λ)·x_tgt_train` (label mengikuti komponen dominan) untuk menjembatani dua distribusi TANPA label target uji. Pembanding untuk few-shot.

**Pembanding tetap:** baseline single-source (0% target, dari T3) dan joint (batas atas, T6).

**Hipotesis jujur:** few-shot kemungkinan naik tajam bahkan dgn sedikit label target → pesan: cross-network NIDS praktis butuh sedikit kalibrasi, dan SFM membuatnya efisien. Jika tidak naik, shift-nya lebih dalam dari sekadar kalibrasi. Semua angka apa adanya.

> **Protokol jujur few-shot:** fraksi label target diambil dari **UNSW train** (arah CIC→UNSW) atau **CIC train** (arah UNSW→CIC) — BUKAN dari test. Uji selalu di test target yang tak tersentuh. Tak ada kebocoran.

> Jalankan di SageMaker (butuh `cleaned_100.pkl` + CSV UNSW).

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn'), ('xgboost','xgboost')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import pickle, os, json
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../domain_adaptation.json'
SEED=42
FRACS=[0.0, 0.01, 0.05, 0.10, 0.25]   # fraksi label target utk few-shot
print('CIC:', os.path.exists(CIC_PKL), '| UNSW tr:', os.path.exists(UNSW_TRAIN), '| te:', os.path.exists(UNSW_TEST))

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())

def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def make_xgb():
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist')

def ev(yt,yp):
    return dict(mcc=float(matthews_corrcoef(yt,yp)),f1=float(f1_score(yt,yp,zero_division=0)),
                acc=float(accuracy_score(yt,yp)),confusion=confusion_matrix(yt,yp).tolist())

In [ ]:
# --- Muat data ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)

unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values

# matriks + z-score per dataset
Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC tr/te:',Xc_tr.shape,Xc_te.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)

In [ ]:
# ============================================================
# EKSPERIMEN 1 — FEW-SHOT TARGET ADAPTATION
# ============================================================
def few_shot(Xs, ys, Xt_tr, yt_tr, Xt_te, yt_te, fracs, tag):
    """latih di source + fraksi label target(train); uji di target(test)."""
    print('='*66); print(f'FEW-SHOT: {tag}'); print('='*66)
    rng=np.random.RandomState(SEED)
    out=[]
    for fr in fracs:
        if fr==0.0:
            Xtr, ytr = Xs, ys
        else:
            n=int(len(Xt_tr)*fr)
            idx=rng.choice(len(Xt_tr), n, replace=False)
            Xtr=np.vstack([Xs, Xt_tr[idx]]); ytr=np.concatenate([ys, yt_tr[idx]])
        m=make_xgb(); m.fit(Xtr,ytr)
        r=ev(yt_te, m.predict(Xt_te))
        n_tgt = 0 if fr==0.0 else int(len(Xt_tr)*fr)
        out.append(dict(frac=fr, n_target=n_tgt, **{k:r[k] for k in ('mcc','f1','acc')}))
        print(f'  frac={fr:.2f} (n_tgt={n_tgt:>6,})  MCC={r["mcc"]:+.4f}  F1={r["f1"]:.4f}  ACC={r["acc"]:.4f}')
    return out

results={}
# arah CIC->UNSW: source=CIC, target=UNSW
results['cic2unsw_fewshot']=few_shot(Xc_tr,yc_tr, Xu_tr,y_utr, Xu_te,y_ute, FRACS, 'latih CIC + x% UNSW -> uji UNSW')
# arah UNSW->CIC: source=UNSW, target=CIC
results['unsw2cic_fewshot']=few_shot(Xu_tr,y_utr, Xc_tr,yc_tr, Xc_te,yc_te, FRACS, 'latih UNSW + x% CIC -> uji CIC')

In [ ]:
# ============================================================
# EKSPERIMEN 2 — CROSS-DATASET MIXUP (tanpa label target uji)
# ============================================================
# x_mix = lam*x_src + (1-lam)*x_tgt_train ; label = label komponen dominan (lam>=0.5 -> src, else tgt).
# tgt_train dipakai HANYA fiturnya utk mixup (label train target dipakai sbg pseudo utk komponen tgt).
def mixup(Xs, ys, Xt, yt, n_mix, alpha=0.4):
    rng=np.random.RandomState(SEED)
    si=rng.choice(len(Xs), n_mix, replace=True)
    ti=rng.choice(len(Xt), n_mix, replace=True)
    lam=rng.beta(alpha, alpha, size=(n_mix,1))
    Xmix=lam*Xs[si]+(1-lam)*Xt[ti]
    ymix=np.where(lam.ravel()>=0.5, ys[si], yt[ti])
    return Xmix, ymix.astype(int)

def run_mixup(Xs, ys, Xt_tr, yt_tr, Xt_te, yt_te, tag):
    print('='*66); print(f'MIXUP: {tag}'); print('='*66)
    n_mix=min(len(Xs), 100000)
    Xmix,ymix=mixup(Xs,ys, Xt_tr,yt_tr, n_mix)
    Xtr=np.vstack([Xs, Xmix]); ytr=np.concatenate([ys, ymix])
    m=make_xgb(); m.fit(Xtr,ytr)
    r=ev(yt_te, m.predict(Xt_te))
    print(f'  MCC={r["mcc"]:+.4f} F1={r["f1"]:.4f} ACC={r["acc"]:.4f}  (n_mix={n_mix:,})')
    return r

# mixup pakai fitur+label train target (bukan test) -> tetap unsupervised thd test target
results['cic2unsw_mixup']=run_mixup(Xc_tr,yc_tr, Xu_tr,y_utr, Xu_te,y_ute, 'CIC + mix(CIC,UNSWtrain) -> uji UNSW')
results['unsw2cic_mixup']=run_mixup(Xu_tr,y_utr, Xc_tr,yc_tr, Xc_te,yc_te, 'UNSW + mix(UNSW,CICtrain) -> uji CIC')

In [ ]:
# --- Ringkasan komparatif ---
def mcc_at(lst, fr): return next(x['mcc'] for x in lst if x['frac']==fr)
print('KURVA FEW-SHOT (MCC vs fraksi label target):')
print(f"{'frac':>6} {'CIC->UNSW':>12} {'UNSW->CIC':>12}")
for fr in FRACS:
    print(f"{fr:>6.2f} {mcc_at(results['cic2unsw_fewshot'],fr):>12.4f} {mcc_at(results['unsw2cic_fewshot'],fr):>12.4f}")
print('\nMIXUP (tanpa label target uji):')
print(f"  CIC->UNSW MCC={results['cic2unsw_mixup']['mcc']:+.4f} | UNSW->CIC MCC={results['unsw2cic_mixup']['mcc']:+.4f}")
print('\nPembanding: baseline single-source = few-shot frac=0.0; joint (T6) = 0.912/0.732 (in-domain ~ atas).')

meta=dict(
  deskripsi='Domain adaptation: few-shot target adaptation + cross-dataset mixup (Model A biner, z-score per dataset).',
  features=CANON, fracs=FRACS,
  fewshot=dict(cic2unsw=results['cic2unsw_fewshot'], unsw2cic=results['unsw2cic_fewshot']),
  mixup=dict(cic2unsw=results['cic2unsw_mixup'], unsw2cic=results['unsw2cic_mixup']),
)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('\nSaved:', OUT_JSON)